# 07 — Circle/sphere fitting and data normalization

**Needed for the assignment:** $\bigstar$ YES. **Study:** 1.5 h, **Practice:** 1 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

The sphere equation $\|\mathbf{p}-\mathbf{c}\|^2=r^2$ is not linear in $(\mathbf{c},r)$. Multiply it out and rename the coefficients, and it becomes **linear** in new unknowns, so the usual "$A\mathbf{h}=\mathbf{0}$, smallest eigenvector" recipe works (an *algebraic* fit). Because the matrix mixes $x^2$ (about $10^6$ in mm) with $1$, **data normalization** is needed first, as for the homography.

## Key formulas

## Circle and Sphere Fitting

### 1. Algebraic Equation

A general sphere in 3D is represented by:

$$
a(x^2+y^2+z^2)+bx+cy+ez+f=0
$$

For each 3D point $(x,y,z)$, construct one row:

$$
\boxed{
[x^2+y^2+z^2,\ x,\ y,\ z,\ 1]
}
$$

### 2. Calculate the Center and Radius

Divide the equation by $a$ and complete the squares.

**Center:**

$$
\boxed{
\mathbf{c}=-\frac{1}{2a}
\begin{bmatrix}
b\\c\\e
\end{bmatrix}
}
$$

**Radius:**

$$
\boxed{
r=\sqrt{\|\mathbf{c}\|^2-\frac{f}{a}}
}
$$

### 3. Distance of a Point to the Circle or Sphere

For a point $\mathbf{p}$, calculate its distance from the circle or sphere surface:

$$
\boxed{
d(\mathbf{p})=
\left|\|\mathbf{p}-\mathbf{c}\|-r\right|
}
$$

This is the absolute difference between the distance to the center and the radius.

### 4. Minimal Number of Points

- **Circle (2D):** 3 non-collinear points.
- **Sphere (3D):** 4 non-coplanar points.

These are the minimum numbers needed to uniquely determine a circle or sphere.

## Data Normalization

Normalization rescales and centers points to improve numerical stability when fitting geometric models.

### 1. Normalize the Points

Calculate the centroid:

$$
\bar{\mathbf{p}}=
\frac{1}{N}\sum_{i=1}^{N}\mathbf{p}_i
$$

Then calculate the scale:

$$
s=
\frac{\sqrt{k}}
{\operatorname{mean}\|\mathbf{p}_i-\bar{\mathbf{p}}\|}
$$

where:
- $k=2$ for 2D points.
- $k=3$ for 3D points.

Transform each point:

$$
\boxed{
\mathbf{q}=s(\mathbf{p}-\bar{\mathbf{p}})
}
$$

The normalized points are centered around the origin and scaled to have a mean distance of $\sqrt{k}$ from the origin.

### 2. Denormalize the Results

After fitting the circle or sphere in normalized coordinates, convert the center and radius back:

$$
\boxed{
\mathbf{c}=\frac{\mathbf{c}_q}{s}+\bar{\mathbf{p}}
}
$$

$$
\boxed{
r=\frac{r_q}{s}
}
$$

where $\mathbf{c}_q$ and $r_q$ are the center and radius in normalized coordinates.

### 3. 2D Homogeneous Normalization Matrix

In 2D, the normalization transformation can be represented using a homogeneous transformation matrix:

$$
\boxed{
T=
\begin{bmatrix}
s&0&-s\bar{u}\\
0&s&-s\bar{v}\\
0&0&1
\end{bmatrix}
}
$$

For a point $\mathbf{p}=[u,v,1]^T$:

$$
\mathbf{q}=T\mathbf{p}
$$

This is the normalization matrix used in Lab 6 / lecture.

# Worked example (on paper): circle through 3 points

Points $(3,2),(1,4),(-1,2)$. Each row is $[x^2+y^2,\ x,\ y,\ 1]$: $$\begin{bmatrix}13&3&2&1\\17&1&4&1\\5&-1&2&1\end{bmatrix}\begin{bmatrix}a\\b\\c\\f\end{bmatrix}=\mathbf{0}$$ **1. Eliminate.** (row 1 $-$ row 3): $8a+4b=0\Rightarrow b=-2a$.  (row 2 $-$ row 3): $12a+2b+2c=0\Rightarrow 12a-4a+2c=0\Rightarrow c=-4a$.

**2.** Row 3: $5a-b+2c+f=0\Rightarrow5a+2a-8a+f=0\Rightarrow f=a$.

**3.** Choose $a=1$: $\mathbf{h}=(1,-2,-4,1)$. (With 3 points the null space is exact; with more points you take the smallest eigenvector.)

**4. Centre and radius.** $\mathbf{c}=-\tfrac12(-2,-4)=(1,2)$,  $r^2=1^2+2^2-1=4\Rightarrow r=2$.   Check $(3,2)$: $\sqrt{(3-1)^2+0}=2$ ✓.

# The same in Python / NumPy (circle, then a noisy half-sphere in mm)

In [2]:
import numpy as np

def smallest_eigvec(M):
    val, vec = np.linalg.eig(M)
    val, vec = val.real, vec.real      # eig may return complex dtype
    return vec[:, np.argmin(val)]

# --- 1) circle through 3 points (paper example) ---
pts = np.array([[3, 2], [1, 4], [-1, 2]], dtype=float)
A = np.column_stack([np.sum(pts**2, axis=1), pts, np.ones(3)])   # [x^2+y^2, x, y, 1]
print("A =\n", A)
h = smallest_eigvec(A.T @ A)
h = h / h[0]                       # scale so that a = 1 (only for readability)
a, b, c, f = h
centre = -np.array([b, c]) / (2 * a)
r = np.sqrt(centre @ centre - f / a)
print("h =", np.round(h, 4), " centre =", np.round(centre, 4), " r =", round(r, 4))

# --- 2) sphere fit with normalization (3D) ---
def normalization(P):
    mean = P.mean(axis=0)
    scale = np.sqrt(3) / np.mean(np.linalg.norm(P - mean, axis=1))
    return mean, scale

def fit_sphere(P, normalize=True):
    mean, scale = normalization(P) if normalize else (np.zeros(3), 1.0)
    Q = (P - mean) * scale
    A = np.column_stack([np.sum(Q**2, axis=1), Q, np.ones(len(Q))])
    a, b, c, e, f = smallest_eigvec(A.T @ A)
    centre_q = -np.array([b, c, e]) / (2 * a)
    r_q = np.sqrt(centre_q @ centre_q - f / a)
    return centre_q / scale + mean, r_q / scale, np.linalg.cond(A.T @ A)

# noisy points on one side of a ball (like a depth camera sees it), in mm
rng = np.random.default_rng(1)
true_c, true_r = np.array([-80.0, -20.0, 1270.0]), 215.0
dirs = rng.normal(size=(500, 3))
dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
dirs = dirs[dirs[:, 2] < 0]                       # only the half facing the camera
P = true_c + true_r * dirs + rng.normal(0, 1.0, size=dirs.shape)

for norm in (False, True):
    c_, r_, cond = fit_sphere(P, norm)
    print(f"normalize={norm!s:5}: centre={np.round(c_, 2)}, r={r_:.2f}, cond(A^T A)={cond:.1e}")

dist = np.abs(np.linalg.norm(P - c_, axis=1) - r_)
print("mean distance to sphere =", round(dist.mean(), 3), "mm")

A =
 [[13.  3.  2.  1.]
 [17.  1.  4.  1.]
 [ 5. -1.  2.  1.]]
h = [ 1. -2. -4.  1.]  centre = [1. 2.]  r = 2.0
normalize=False: centre=[ -79.8   -19.75 1270.18], r=215.00, cond(A^T A)=2.7e+19
normalize=True : centre=[ -79.8   -19.75 1270.19], r=215.01, cond(A^T A)=1.3e+05
mean distance to sphere = 0.771 mm


The condition number drops from about $10^{19}$ to about $10^{5}$ with normalization. Without it, the result depends on luck with rounding: here it still works in 64-bit floats, but small samples, `float32` data, or a far-away ball can break it.

## Where this is used in the assignment

`fit_sphere` in `ransac.py` is this exact code. It is called with 4 points inside RANSAC, then with all inliers to refine. `distance_point_sphere` is $|\|\mathbf{p}-\mathbf{c}\|-r|$. A fit with $a\approx0$ means the points are (almost) on a plane, so it is rejected.

## Course / lab link

Lecture 00 lists "Circle/Sphere fitting" under point cloud processing, and the RANSAC slides give $m=4$ for a sphere. The normalization is the same as in lec01 (homography, projection matrix) and Lab 6 Task 2.

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

Circle through $(4,1),(2,3),(0,1)$ by hand (eliminate as in the example). Check one point.

> **Expected result:** Centre $(2,1)$, $r=2$.

In [3]:
# Exercise 1: your solution


### Exercise 2

Compute the normalization ($\bar{\mathbf p}$, $s$, $T$) for the 2D points $(0,0),(2,0),(2,2),(0,2)$ by hand. What do the normalized points look like?

> **Expected result:** $\bar{\mathbf p}=(1,1)$, every distance is $\sqrt2$, so $s=1$ and $T=\begin{bmatrix}1&0&-1\\0&1&-1\\0&0&1\end{bmatrix}$. Normalized points $(\pm1,\pm1)$.

In [4]:
# Exercise 2: your solution


### Exercise 3

Fit a sphere (with normalization) to the provided clean points `Ps`. Then fit it again with the 20 outliers `out` added. What happens?

> **Expected result:** Clean: centre $\approx(1.0012,2.0007,3.0012)$, $r\approx0.5014$. With outliers: centre $\approx(1.135,2.010,2.960)$, $r\approx0.762$. Twenty bad points ruin least squares, which is why we need RANSAC.

In [5]:
# --- provided data (run me first) ---
# Ps  : (200, 3) points on a sphere, centre (1, 2, 3), radius 0.5, noise std 0.01
# out : (20, 3) random outliers inside the box [0,2] x [1,3] x [2,4]
rng = np.random.default_rng(7)
d = rng.normal(size=(200, 3)); d /= np.linalg.norm(d, axis=1, keepdims=True)
Ps = np.array([1, 2, 3.]) + 0.5 * d + rng.normal(0, 0.01, (200, 3))
out = rng.uniform([0, 1, 2], [2, 3, 4], (20, 3))

In [6]:
# Exercise 3: your solution


### Exercise 4

Why are 4 points on one plane a bad sample for a sphere? Fit the points $(0,0,0),(1,0,0),(0,1,0),(1,1,0)$ and look at the eigenvalues of $A^TA$.

> **Expected result:** The two smallest eigenvalues are both $\approx0$: the solution is not unique (infinitely many spheres, or a plane with $a=0$), so the sample must be rejected.

In [7]:
# Exercise 4: your solution
